# NB4 — MobileViT + LIFE — TopoDistil++

The architecture-generalization experiment ("Recommended new structure" / Section 16, `TopoDistil_bulletproof.md`): does topology-guided spatial supervision (LIFE) transfer from an efficient CNN (RepViT, NB2) to an efficient CNN–Transformer hybrid (MobileViT)?

This notebook does **not** re-derive anything NB2 or NB3 already established. It reuses:
- NB1's topology data (`A_gauss_maps.h5`, `persistence_diagrams.pkl`, `patch_meta.csv`, `splits.json`) directly.
- NB2's `ResidualProjector` / `TopoGate` / `TopoAlignHead` / `build_A_proj` / `shuffle_spatial_per_sample` classes and functions, unchanged.
- NB3's resolved backbone name and CNN/Transformer injection-layer path, loaded from its output rather than re-probed here.

```
NB1 (topology)      NB2 (RepViT, done)      NB3 (MobileViT sanity, done)
      |                                             |
      +---------------------------------------------+
                          |
                          v
                    NB4 (this notebook)
                MobileViT + LIFE
                          |
                          v
                    NB5: cross-backbone analysis
```


## 0. Setup

In [ ]:
!pip install -q timm ripser scikit-image


In [ ]:
import os, sys, json, time, pickle, random, math
from pathlib import Path

import numpy as np
import pandas as pd
import h5py
import torch
import torch.nn as nn
import torch.nn.functional as F_
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, accuracy_score
from tqdm.auto import tqdm

import timm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cpu":
    print("WARNING: no GPU detected. This notebook trains 8 real runs (see Section 12) -- "
          "switch to a GPU runtime (Settings > Accelerator > GPU T4 x2) before running Section 13.")


## 1. Implementation assumptions

1. **Scope.** Originally 4 configurations, per `TopoDistil_bulletproof.md`'s NB4 plan -- `mobilevit_baseline`, `mobilevit_life`, `mobilevit_shuffled_control`, `mobilevit_oracle`. This is deliberately not a repeat of NB2's 13-row ablation battery: NB2 already establishes *whether LIFE works* (the mechanism question); NB4 only asks *whether it transfers* to a different backbone family. Re-running the full battery here would roughly double the project's total compute for a question the 4-row matrix already answers. **Update:** a 5th configuration, `mobilevit_lattn_only_no_gate` (gate-free, distillation-only -- `use_L_attn=True`, `gate_active=False`), was added after NB2's equivalent row (`row4_lattn_only_no_gate`) outperformed full LIFE on RepViT. This is a genuinely new arm for MobileViT (not part of the original `TopoDistil_bulletproof.md` NB4 plan), added to test whether the same "gate isn't the part that's helping" pattern transfers, rather than being read off RepViT alone.
2. **Seeds.** `mobilevit_baseline`, `mobilevit_life`, and (as of the update above) `mobilevit_lattn_only_no_gate` get `SEEDS_CORE` (3 seeds each) since these are the arms NB5's statistical comparisons are built from. `mobilevit_shuffled_control` and `mobilevit_oracle` get 1 seed each -- they're sanity/upper-bound checks, not the comparison itself. Total: 3+3+3+1+1 = 11 runs.
3. **Gate form.** `gate_form="B"` (residual gate) throughout, matching NB2's own choice for every row past the ablation stage (`row10_full_form_b` onward, including its shuffled/oracle controls) -- so NB4's shuffled/oracle controls are testing the same gate formulation NB2 already validated, not a fresh, unvalidated one.
4. **Backbone and injection layer are not re-derived here.** Both come straight from NB3's `mobilevit_validation_summary.json` (Section 4 below) -- the same backbone NB3 already shape/param/memory/speed-validated, hooked at the same CNN/Transformer boundary NB3 already located. If NB3 fell back to `mobilevit_xs` or `mobilevit_xxs` (its GPU-memory probe, Section 7), NB4 automatically follows that choice, and NB5's cross-backbone comparison should note if the resolved backbone isn't `mobilevit_s` -- see NB3's own note on this.
5. **Reused model code.** `ResidualProjector`, `build_A_proj`, `TopoGate`, `TopoAlignHead`, `shuffle_spatial_per_sample`, `PCamTopoDataset`, `augment_pair`, and the oracle on-the-fly-PH pipeline (`compute_oracle_maps` and its helpers) are copied from NB2 **unchanged** -- same class definitions, same forward-hook mechanism, same shuffled-control semantics (permutes `A_proj` post-residual/post-homology-selection, not the raw prior). `TopKDEmbeddingBaseline` and `sobel_map` (NB2's competing-baseline machinery for `row2`/`row3`) are dropped here -- NB4's matrix has no attention-transfer or TopKD rows, so keeping them would just be dead code.
6. **No PH at inference, preserved.** `evaluate(..., use_topology=False)` is used for every `standard_val`/`standard_test` metric -- the gate is bypassed and the backbone runs exactly as it would without LIFE, matching the paper's efficiency claim. Only the oracle row's test-set evaluation recomputes topology on the fly (`use_topology=True, on_the_fly_topology=True`), and that's an explicit upper-bound probe, not the deployment path.
7. **`H_THRESH` is recomputed here, not loaded from NB2.** It's the median summed train-set entropy from NB1's `patch_meta.csv` (Section 3 below) -- since NB2 and NB4 read the exact same `patch_meta.csv` off the exact same fixed split, this reproduces NB2's value exactly; recomputing it locally just avoids adding NB2's output as a third required input.


## 2. Config

In [ ]:
CONFIG = {
    "NB1_DIR": "/kaggle/input/notebooks/claudeisnotclaude/data-setup",                # topology data -- auto-discovered in Section 3
    "NB3_DIR": "/kaggle/input/notebooks/claudeisnotclaude/mobilevit-backbone-val",                # MobileViT backbone validation -- auto-discovered in Section 4
    "PCAM_DIR": "/kaggle/input/datasets/andrewmvd/metastatic-tissue-classification-patchcamelyon",

    "BACKBONE": None,               # filled in from NB3's summary (Section 4) -- not re-derived
    "INJECTION_LAYER": None,        # filled in from NB3's summary (Section 4) -- not re-derived
    "IMG_SIZE": 96,

    "BATCH_SIZE": 32,               # matches NB2/NB3
    "LR": 1e-3,                     # peak LR for newly-initialized modules (gate/residual/align head)
    "LR_BACKBONE_MULT": 0.1,        # pretrained backbone gets LR * this (discriminative LR, Section 11 fix)
    "WARMUP_EPOCHS": 1,             # linear warmup over the first epoch, then cosine decay
    "MIN_LR_FRAC": 0.05,            # cosine decay floor, as a fraction of peak LR (never fully to 0)
    "EPOCHS": 12,                   # matches NB2's real epoch budget
    "LAMBDA": 0.5,                  # weight on L_attn, matches NB2 (Section 3.7)

    "VAL_SUBSET": 5000,             # matches NB2
    "TEST_SUBSET": 5000,
    "ORACLE_SUBSET": 1000,          # oracle eval needs on-the-fly PH -> keep small, matches NB2

    "SEEDS_CORE": [0, 1, 2],        # mobilevit_baseline, mobilevit_life, mobilevit_lattn_only_no_gate
    "SEED_SINGLE": 0,               # mobilevit_shuffled_control, mobilevit_oracle

    "OUT_DIR": "/kaggle/working/mobilevit_life_results",
    "SAVE_WEIGHTS_FOR": {"mobilevit_baseline", "mobilevit_life", "mobilevit_oracle",
                          "mobilevit_lattn_only_no_gate"},  # added: gate-free distillation-only arm
}
os.makedirs(CONFIG["OUT_DIR"], exist_ok=True)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

CONFIG


## 3. Load NB1 checkpoint (topology data)

Unlike NB3, this notebook *does* need NB1's topology outputs -- `A_gauss_maps.h5`, `persistence_diagrams.pkl`, and `patch_meta.csv` -- since the whole point of NB4 is to train MobileViT with LIFE (topology gating) active. Auto-discovered the same robust way as NB3: search `/kaggle/input` for `splits.json` rather than a hardcoded, account-specific path.

In [ ]:
def find_nb1_checkpoint(search_root="/kaggle/input"):
    root = Path(search_root)
    if not root.exists():
        raise FileNotFoundError(f"Kaggle input directory not found: {root}")
    matches = list(root.rglob("splits.json"))
    for f in matches:
        if f.parent.name.lower() == "topology_checkpoint":
            return f.parent
    if len(matches) == 1:
        return matches[0].parent
    return None


nb1_dir = find_nb1_checkpoint()
if nb1_dir is None:
    print("ERROR: Could not find NB1 topology checkpoint.")
    print("\nAll splits.json files found under /kaggle/input:")
    for f in Path("/kaggle/input").rglob("splits.json"):
        print("  ", f)
    raise FileNotFoundError(
        "\nNB1 output was not found. Make sure it's attached via "
        "Kaggle \u2192 Add Input \u2192 Notebook Output.")

CONFIG["NB1_DIR"] = str(nb1_dir)
print("\u2713 NB1 checkpoint found:", CONFIG["NB1_DIR"])

with open(f'{CONFIG["NB1_DIR"]}/splits.json') as f:
    splits = json.load(f)
with open(f'{CONFIG["NB1_DIR"]}/config_used.json') as f:
    nb1_config = json.load(f)

patch_meta = pd.read_csv(f'{CONFIG["NB1_DIR"]}/patch_meta.csv').set_index("patch_idx")

with open(f'{CONFIG["NB1_DIR"]}/persistence_diagrams.pkl', "rb") as f:
    diagrams_store = pickle.load(f)  # not directly used downstream; loaded for parity with NB1's output / future debugging

h5_maps_path = f'{CONFIG["NB1_DIR"]}/A_gauss_maps.h5'
with h5py.File(h5_maps_path, "r") as f:
    map_patch_order = np.array(f["patch_idx"])
patch_to_row = {int(p): i for i, p in enumerate(map_patch_order)}

train_idx = np.array(splits["train_subsample_idx"])
train_labels = np.array(splits["train_subsample_labels"])
H_THRESH = float(np.median(patch_meta.loc[train_idx, ["entropy_h0", "entropy_h1"]].sum(axis=1)))
CONFIG["H_THRESH"] = H_THRESH

print(f"Train subsample: {len(train_idx)} patches")
print(f"H_thresh (median summed entropy): {H_THRESH:.3f}")


from pathlib import Path

def find_pcam_files(root):
    root = Path(root)

    found = {}

    # Image HDF5 files in this Kaggle dataset
    image_names = {
        "train_x": "training_split.h5",
        "valid_x": "validation_split.h5",
        "test_x": "test_split.h5",
    }

    # Label HDF5 files
    label_names = {
        "train_y": "camelyonpatch_level_2_split_train_y.h5",
        "valid_y": "camelyonpatch_level_2_split_valid_y.h5",
        "test_y": "camelyonpatch_level_2_split_test_y.h5",
    }

    all_h5 = list(root.rglob("*.h5"))

    for key, filename in {**image_names, **label_names}.items():
        matches = [f for f in all_h5 if f.name == filename]
        if matches:
            found[key] = matches[0]

    return found


pcam_files = find_pcam_files(CONFIG["PCAM_DIR"])

required = [
    "train_x", "train_y",
    "valid_x", "valid_y",
    "test_x", "test_y"
]

missing = [k for k in required if k not in pcam_files]

if missing:
    print("Could not find:", missing)
    print("\nH5 files found:")
    for f in sorted(Path(CONFIG["PCAM_DIR"]).rglob("*.h5")):
        print(" ", f)
else:
    print("Found all PCam files:")
    for k, v in pcam_files.items():
        print(f"  {k}: {v}")

## 4. Load NB3 checkpoint (resolved backbone + injection layer)

Pulls `CONFIG["BACKBONE"]` and `CONFIG["INJECTION_LAYER"]` directly from NB3's saved summary instead of re-deriving them -- NB3 already shape-validated, size-matched against RepViT, GPU-memory-probed, and located the CNN/Transformer boundary for this exact backbone. Re-running that logic here would risk silently landing on a *different* injection point than the one NB3 actually validated.

In [ ]:
def find_nb3_checkpoint(search_root="/kaggle/input"):
    root = Path(search_root)
    if not root.exists():
        raise FileNotFoundError(f"Kaggle input directory not found: {root}")
    matches = list(root.rglob("mobilevit_validation_summary.json"))
    if len(matches) == 1:
        return matches[0].parent
    if len(matches) > 1:
        print("Multiple mobilevit_validation_summary.json found -- using the first:")
        for m in matches:
            print("  ", m)
        return matches[0].parent
    return None


nb3_dir = find_nb3_checkpoint()
if nb3_dir is None:
    raise FileNotFoundError(
        "Could not find NB3's mobilevit_validation_summary.json under /kaggle/input. "
        "Make sure NB3's output is attached via Kaggle \u2192 Add Input \u2192 Notebook Output.")

CONFIG["NB3_DIR"] = str(nb3_dir)
print("\u2713 NB3 checkpoint found:", CONFIG["NB3_DIR"])

with open(f'{CONFIG["NB3_DIR"]}/mobilevit_validation_summary.json') as f:
    nb3_summary = json.load(f)

CONFIG["BACKBONE"] = nb3_summary["backbone"]
CONFIG["INJECTION_LAYER"] = nb3_summary["injection_layer"]
assert nb3_summary["img_size"] == CONFIG["IMG_SIZE"], (
    f"NB3 validated at img_size={nb3_summary['img_size']}, NB4 configured for "
    f"{CONFIG['IMG_SIZE']} -- these must match.")

print(f"Resolved backbone:       {CONFIG['BACKBONE']}")
print(f"Resolved injection layer: {CONFIG['INJECTION_LAYER']}")
if CONFIG["BACKBONE"] != "mobilevit_s":
    print(f"\nNOTE: NB3's GPU-memory probe fell back to {CONFIG['BACKBONE']} instead of the "
          "default mobilevit_s candidate. NB5's cross-backbone comparison against RepViT "
          "(~4.7M params) should flag this parameter-count mismatch explicitly.")

n_params = nb3_summary["param_counts"].get(CONFIG["BACKBONE"])
flops = nb3_summary.get("flops", {}).get(CONFIG["BACKBONE"])
flops_method = nb3_summary.get("flops_method", {}).get(CONFIG["BACKBONE"])
print(f"\n(from NB3, not recomputed) params={n_params/1e6:.2f}M   "
      f"flops={('%.1fM' % (flops/1e6)) if flops else 'n/a'} (method={flops_method})")


## 5. Dataset

`PCamTopoDataset` and `augment_pair`, reused **unchanged** from NB2 -- same lazy h5 handle pattern, same aligned/unaligned augmentation modes, same per-patch topology-map lookup via `patch_to_row`.

In [ ]:
def augment_pair(img, amap, mode, rng):
    """img: (H,W,3) uint8-like float array. amap: (2,H,W) float array."""
    k = rng.integers(0, 4)
    flip = rng.integers(0, 2)

    img_t = np.rot90(img, k, axes=(0, 1))
    if flip:
        img_t = np.flip(img_t, axis=1)

    if mode == "aligned":
        amap_t = np.rot90(amap, k, axes=(1, 2))
        if flip:
            amap_t = np.flip(amap_t, axis=2)
    else:  # unaligned: independent random transform on the map (unused by NB4's matrix, kept for parity with NB2)
        k2 = rng.integers(0, 4)
        flip2 = rng.integers(0, 2)
        amap_t = np.rot90(amap, k2, axes=(1, 2))
        if flip2:
            amap_t = np.flip(amap_t, axis=2)

    return np.ascontiguousarray(img_t), np.ascontiguousarray(amap_t)


class PCamTopoDataset(Dataset):
    def __init__(self, indices, split, labels=None, augment=True, aug_mode="aligned", seed=0):
        self.indices = np.asarray(indices)
        self.split = split
        self.labels = labels
        self.augment = augment
        self.aug_mode = aug_mode
        self.rng = np.random.default_rng(seed)
        self._h5 = None
        self._h5_maps = None

    def _images(self):
        if self._h5 is None:
            self._h5 = h5py.File(pcam_files[f"{self.split}_x"], "r")
            self._x_key = list(self._h5.keys())[0]
        return self._h5[self._x_key]

    def _maps(self):
        if self._h5_maps is None:
            self._h5_maps = h5py.File(h5_maps_path, "r")
        return self._h5_maps["A_gauss"]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = int(self.indices[i])
        img = np.array(self._images()[idx]).astype(np.float32) / 255.0

        if self.split == "train" and idx in patch_to_row:
            row = patch_to_row[idx]
            amap = np.array(self._maps()[row]).astype(np.float32)
            entropy = patch_meta.loc[idx, ["entropy_h0", "entropy_h1"]].to_numpy(dtype=np.float32)
        else:
            amap = np.zeros((2, CONFIG["IMG_SIZE"], CONFIG["IMG_SIZE"]), dtype=np.float32)
            entropy = np.zeros(2, dtype=np.float32)

        if self.augment:
            img, amap = augment_pair(img, amap, self.aug_mode, self.rng)

        label = int(self.labels[i]) if self.labels is not None else -1
        img_t = torch.from_numpy(img.transpose(2, 0, 1).copy()).float()
        amap_t = torch.from_numpy(amap.copy()).float()
        return img_t, amap_t, entropy, label, idx


## 6. Model components

`ResidualProjector`, `build_A_proj`, `TopoGate`, `TopoAlignHead`, `shuffle_spatial_per_sample` -- reused **unchanged** from NB2. Dropped `TopKDEmbeddingBaseline` and `sobel_map`: those only back NB2's `row2_attention_transfer`/`row3_topkd_style` competing-baseline rows, which aren't part of NB4's 4-row matrix (Section 1, point 5).

In [ ]:
class ResidualProjector(nn.Module):
    """f_theta: learned residual on top of the fixed Gaussian prior (Section 3.4, Step 2)."""
    def __init__(self, channels=2, hidden=16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(channels, hidden, 3, padding=1), nn.BatchNorm2d(hidden), nn.ReLU(inplace=True),
            nn.Conv2d(hidden, channels, 3, padding=1),
        )

    def forward(self, a_gauss):
        return self.net(a_gauss)


def build_A_proj(a_gauss, residual_projector, use_residual, homology_mode):
    """A_proj = A_gauss + f_theta(A_gauss), with H0/H1 channel selection."""
    if use_residual:
        a_proj = a_gauss + residual_projector(a_gauss)
    else:
        a_proj = a_gauss

    if homology_mode == "h0":
        a_proj = torch.stack([a_proj[:, 0], torch.zeros_like(a_proj[:, 0])], dim=1)
    elif homology_mode == "h1":
        a_proj = torch.stack([torch.zeros_like(a_proj[:, 1]), a_proj[:, 1]], dim=1)
    return a_proj


class TopoGate(nn.Module):
    """Section 3.5. form='A' -> multiplicative dual gate. form='B' -> residual gate (used throughout NB4)."""
    def __init__(self, form="B"):
        super().__init__()
        self.form = form
        self.gate_combine = nn.Conv2d(2, 1, kernel_size=1)
        self.beta = nn.Parameter(torch.tensor(1.0))
        self.alpha = nn.Parameter(torch.tensor(0.0))
        self.last_gate_mean = None

    def forward(self, feat, a_proj, entropy, h_thresh):
        a_resized = F_.interpolate(a_proj, size=feat.shape[-2:], mode="bilinear", align_corners=False)
        combined = self.gate_combine(a_resized)
        gate = torch.sigmoid(self.beta * combined + self.alpha)
        entropy_sum = entropy.sum(dim=1) if entropy.dim() > 1 else entropy
        gamma = torch.sigmoid(entropy_sum - h_thresh).view(-1, 1, 1, 1)
        self.last_gate_mean = float((gate * gamma).mean().detach().cpu())

        if self.form == "A":
            return feat * gate * gamma
        else:  # form B
            return feat + feat * gate * gamma


class TopoAlignHead(nn.Module):
    """Projects F down to a 2-channel spatial map for L_attn (Section 3.7)."""
    def __init__(self, in_channels):
        super().__init__()
        self.proj = nn.Conv2d(in_channels, 2, kernel_size=1)

    def forward(self, feat):
        return self.proj(feat)


def shuffle_spatial_per_sample(a_proj):
    """Shuffled control: per-sample random pixel permutation of A_proj, applied AFTER the
    residual projector and homology-mode selection so it operates on the exact map used for
    gating/L_attn. Same permutation across both channels of a sample, so H0/H1 stay
    co-registered with each other (just not with the image); preserves each sample's
    per-channel value histogram while destroying spatial correctness."""
    b, c, h, w = a_proj.shape
    flat = a_proj.reshape(b, c, h * w)
    out = torch.empty_like(flat)
    for i in range(b):
        perm = torch.randperm(h * w, device=a_proj.device)
        out[i] = flat[i][:, perm]
    return out.reshape(b, c, h, w)


## 7. TopoDistilModel (hooked onto NB3's injection layer)

Same hook mechanism as NB2 -- a forward hook on `CONFIG["INJECTION_LAYER"]` captures the feature map and, when the gate is active, replaces the hooked module's output with the gated version before it flows into the next stage. For MobileViT that next stage is the first `MobileVitBlock` (tokenization + Transformer), so a gated feature map here is exactly the doc's "topology gate sits right before tokenization" design. Simplified from NB2's version: no `baseline_type` branch (`topkd`/`attention_transfer`), since NB4's matrix never uses it.

In [ ]:
class TopoDistilModel(nn.Module):
    def __init__(self, backbone_name, gate_active, gate_form, use_L_attn,
                 use_residual, homology_mode, shuffled_control=False):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=True, num_classes=2)
        self.gate_active = gate_active
        self.use_L_attn = use_L_attn
        self.use_residual = use_residual
        self.homology_mode = homology_mode
        self.shuffled_control = shuffled_control

        self.residual_projector = ResidualProjector() if use_residual else None
        self.gate = TopoGate(form=gate_form) if gate_active else None

        self.align_head = None      # lazy-init once we know F's channel count
        self.use_topology = True    # flipped off for standard eval, on for oracle eval

        self._captured_feat = {}
        self._current_a_proj = None
        self._hook_handle = self.backbone.get_submodule(CONFIG["INJECTION_LAYER"]) \
                                          .register_forward_hook(self._hook)

    def _hook(self, module, inp, out):
        self._captured_feat["F"] = out
        if not self.gate_active or not self.use_topology:
            return out
        a_proj = build_A_proj(self._current_a_gauss, self.residual_projector,
                               self.use_residual, self.homology_mode)
        if self.shuffled_control:
            a_proj = shuffle_spatial_per_sample(a_proj)
        self._current_a_proj = a_proj
        out = self.gate(out, a_proj, self._current_entropy, CONFIG["H_THRESH"])
        return out

    def forward(self, img, a_gauss, entropy):
        self._current_a_gauss = a_gauss
        self._current_entropy = entropy
        self._current_a_proj = None
        logits = self.backbone(img)
        feat = self._captured_feat["F"]

        if self.align_head is None and self.use_L_attn:
            self.align_head = TopoAlignHead(feat.shape[1]).to(img.device)

        return logits, feat


## 8. Loss computation

Simplified from NB2's `compute_losses`: only the `L_attn` branch remains (no `attention_transfer`/`topkd` target branches, since those baseline types don't exist in NB4's matrix). Still reuses the cached `model._current_a_proj` from the hook when the gate was active this forward pass, for the same reason NB2 does -- `build_A_proj` runs `residual_projector`, which contains a `BatchNorm2d`; calling it a second time here would give it a second running-stats update per optimizer step, and for the shuffled control would produce a *different* shuffled map than the one the gate actually saw.

In [ ]:
def compute_losses(model, logits, feat, a_gauss, entropy, labels, lam, cfg_):
    ce = F_.cross_entropy(logits, labels)
    total = ce
    l_attn_value = None

    if cfg_["use_L_attn"] and lam > 0:
        a_student = model.align_head(feat)
        if model._current_a_proj is not None:
            # gate was active this forward pass -- reuse exactly what it saw (see markdown above)
            target = model._current_a_proj
        else:
            # gate_active=False, use_L_attn=True: mobilevit_lattn_only_no_gate. Gate was never
            # applied this forward pass, so _current_a_proj is still None here -- build the
            # (full, residual-projected) target directly, matching NB2's row4 semantics exactly.
            target = build_A_proj(a_gauss, model.residual_projector, cfg_["use_residual"],
                                   cfg_["homology_mode"])
        target_resized = F_.interpolate(target, size=a_student.shape[-2:], mode="bilinear",
                                         align_corners=False)
        l_attn = F_.mse_loss(a_student, target_resized)
        total = total + lam * l_attn
        l_attn_value = float(l_attn.detach().cpu())

    return total, ce, l_attn_value


## 9. Oracle on-the-fly topology (test-time PH)

Identical to NB2's pipeline -- same nuclei-extraction / H0 / H1 / Gaussian-map / entropy functions, kept in sync manually (see NB1 for the canonical, documented version). Needed because the oracle row's whole point is: *if PH were computed at test time too (which the deployed model never does), how much further could LIFE push MobileViT?* -- an upper bound, not a claim about the deployed model.

In [ ]:
from skimage.color import rgb2hed
from skimage.filters import threshold_otsu
from skimage.measure import label as sklabel, regionprops
from skimage.morphology import remove_small_objects, binary_opening, disk
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import minimum_spanning_tree
from scipy.spatial.distance import pdist, squareform
from ripser import ripser as ripser_fn

def _extract_nuclei_centroids(patch_rgb, min_area=8):
    hed = rgb2hed(patch_rgb)
    h_channel = hed[:, :, 0]
    h_channel = (h_channel - h_channel.min()) / (np.ptp(h_channel) + 1e-8)
    try:
        thresh = threshold_otsu(h_channel)
    except ValueError:
        return np.zeros((0, 2))
    mask = h_channel > thresh
    mask = binary_opening(mask, footprint=disk(1))
    mask = remove_small_objects(mask, min_size=min_area)
    lbl = sklabel(mask)
    props = regionprops(lbl)
    return np.array([[p.centroid[1], p.centroid[0]] for p in props])

def _h0(centroids):
    n = len(centroids)
    if n < 2:
        return np.zeros(0), np.zeros((0, 2))
    dmat = squareform(pdist(centroids))
    mst = minimum_spanning_tree(csr_matrix(dmat)).tocoo()
    crit_xy = (centroids[mst.row] + centroids[mst.col]) / 2.0
    return mst.data, crit_xy

def _h1(centroids, dmat=None):
    n = len(centroids)
    if n < 3:
        return np.zeros(0), np.zeros((0, 2))
    dgm1 = ripser_fn(centroids, maxdim=1)["dgms"][1]
    dgm1 = dgm1[np.isfinite(dgm1[:, 1])]
    if len(dgm1) == 0:
        return np.zeros(0), np.zeros((0, 2))
    if dmat is None:
        dmat = squareform(pdist(centroids))
    iu = np.triu_indices(n, k=1)
    pair_dists = dmat[iu]
    crit_xy = np.zeros((len(dgm1), 2))
    for k, (b, d) in enumerate(dgm1):
        j = np.argmin(np.abs(pair_dists - b))
        p, q = iu[0][j], iu[1][j]
        crit_xy[k] = (centroids[p] + centroids[q]) / 2.0
    return dgm1[:, 1] - dgm1[:, 0], crit_xy

def _gauss_map(crit_xy, persistences, size, sigma=6.0):
    A = np.zeros((size, size), dtype=np.float32)
    if len(crit_xy) == 0:
        return A
    yy, xx = np.mgrid[0:size, 0:size]
    for (x, y), p in zip(crit_xy, persistences):
        if p <= 0:
            continue
        A += p * np.exp(-((xx - x) ** 2 + (yy - y) ** 2) / (2 * sigma ** 2))
    return A

def _persistence_entropy(persistences):
    """H_i = -sum (p_i/P) log(p_i/P), P = sum p_i. Matches NB1's definition exactly."""
    persistences = persistences[persistences > 0]
    if len(persistences) == 0:
        return 0.0
    P = persistences.sum()
    probs = persistences / P
    return float(-(probs * np.log(probs + 1e-12)).sum())

def compute_oracle_maps(img_batch):
    """img_batch: (B,3,H,W) tensor in [0,1]. Returns (amap, entropy): amap is (B,2,H,W),
    entropy is (B,2) = [entropy_h0, entropy_h1], both computed together from the same
    on-the-fly PH pass (not left at the dataset's train-only placeholder)."""
    imgs = (img_batch.permute(0, 2, 3, 1).cpu().numpy())
    size = CONFIG["IMG_SIZE"]
    out = np.zeros((len(imgs), 2, size, size), dtype=np.float32)
    entropy = np.zeros((len(imgs), 2), dtype=np.float32)
    for i, im in enumerate(imgs):
        centroids = _extract_nuclei_centroids(im)
        if len(centroids) < 3:
            continue
        p0, xy0 = _h0(centroids)
        p1, xy1 = _h1(centroids)
        out[i, 0] = _gauss_map(xy0, p0, size)
        out[i, 1] = _gauss_map(xy1, p1, size)
        entropy[i, 0] = _persistence_entropy(p0)
        entropy[i, 1] = _persistence_entropy(p1)
    return torch.from_numpy(out), torch.from_numpy(entropy)


## 10. Validation / test subsets

In [ ]:
def load_labels(split, n):
    with h5py.File(pcam_files[f"{split}_y"], "r") as f:
        y_key = list(f.keys())[0]
        return np.array(f[y_key]).reshape(-1)[:n]

rng = np.random.default_rng(0)
n_valid_full = splits["n_valid"]
n_test_full = splits["n_test"]

y_valid_full = load_labels("valid", n_valid_full)
y_test_full = load_labels("test", n_test_full)

val_idx = rng.choice(n_valid_full, min(CONFIG["VAL_SUBSET"], n_valid_full), replace=False)
test_idx = rng.choice(n_test_full, min(CONFIG["TEST_SUBSET"], n_test_full), replace=False)
val_labels_sub = y_valid_full[val_idx]
test_labels_sub = y_test_full[test_idx]

print(f"Val subset: {len(val_idx)} | Test subset: {len(test_idx)}")


## 11. Train / eval loops

In [ ]:
# ---------------------------------------------------------------------------
# Section 11 fix: LR schedule (warmup + cosine), discriminative LR, per-epoch
# best-checkpoint selection, and a resumable epoch-level checkpoint system.
# Mirrors the NB2 fix exactly (same root-cause diagnosis, same three changes).
# ---------------------------------------------------------------------------

def run_id(cfg_, seed):
    return f"{cfg_['name']}__seed{seed}"


def make_optimizer(model):
    """Two param groups: pretrained backbone at a lower LR, freshly-initialized
    modules (gate / residual projector / align head) at the full LR. align_head
    must already exist (see the dummy forward pass in train_one_run) so every
    param group is known up front -- no runtime add_param_group, which would
    otherwise desync the LR scheduler's per-group state."""
    backbone_params = list(model.backbone.parameters())
    new_params = []
    if model.gate is not None:
        new_params += list(model.gate.parameters())
    if model.residual_projector is not None:
        new_params += list(model.residual_projector.parameters())
    if model.align_head is not None:
        new_params += list(model.align_head.parameters())

    backbone_lr = CONFIG["LR"] * CONFIG["LR_BACKBONE_MULT"]
    param_groups = [
        {"params": backbone_params, "lr": backbone_lr, "initial_lr": backbone_lr, "name": "backbone"},
    ]
    if new_params:
        param_groups.append(
            {"params": new_params, "lr": CONFIG["LR"], "initial_lr": CONFIG["LR"], "name": "new_modules"}
        )
    return torch.optim.Adam(param_groups)


def make_scheduler(optimizer, steps_per_epoch, epochs):
    """1-epoch linear warmup, then cosine decay to MIN_LR_FRAC of peak over the
    remaining epochs. Same multiplicative factor is applied to every param
    group, so the backbone/new-modules LR ratio set in make_optimizer is
    preserved throughout training."""
    warmup_steps = max(1, steps_per_epoch * CONFIG["WARMUP_EPOCHS"])
    total_steps = max(warmup_steps + 1, steps_per_epoch * epochs)
    min_frac = CONFIG["MIN_LR_FRAC"]

    def lr_lambda(step):
        if step < warmup_steps:
            return float(step + 1) / float(warmup_steps)
        progress = (step - warmup_steps) / float(total_steps - warmup_steps)
        progress = min(max(progress, 0.0), 1.0)
        cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
        return min_frac + (1.0 - min_frac) * cosine

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


@torch.no_grad()
def evaluate(model, loader, use_topology, on_the_fly_topology=False):
    model.eval()
    model.use_topology = use_topology
    all_probs, all_labels = [], []

    for img, amap, entropy, labels, idx in loader:
        img, amap, entropy, labels = (img.to(DEVICE), amap.to(DEVICE),
                                       entropy.to(DEVICE).float(), labels.to(DEVICE))
        if on_the_fly_topology:
            amap, entropy = compute_oracle_maps(img)
            amap, entropy = amap.to(DEVICE), entropy.to(DEVICE).float()
        logits, _ = model(img, amap, entropy)
        probs = F_.softmax(logits, dim=1)[:, 1]
        all_probs.append(probs.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

    all_probs = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)
    auc = roc_auc_score(all_labels, all_probs)
    acc = accuracy_score(all_labels, all_probs > 0.5)
    return {"auc": auc, "acc": acc}


# ---- checkpoint helpers (atomic writes, full resume state) ----------------

def _rng_state():
    state = {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state["cuda"] = torch.cuda.get_rng_state_all()
    return state


def _set_rng_state(state):
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"])
    if torch.cuda.is_available() and "cuda" in state:
        torch.cuda.set_rng_state_all(state["cuda"])


def _cpu_state_dict(model):
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


def _atomic_torch_save(payload, path):
    """Write to a temp file then rename -- rename is atomic on POSIX, so a
    process killed mid-write can never leave a half-written (corrupt)
    checkpoint sitting at `path`."""
    tmp_path = path + ".tmp"
    torch.save(payload, tmp_path)
    os.replace(tmp_path, path)


def save_run_checkpoint(path, epoch, model, optimizer, scheduler, best_val_auc,
                         best_state, best_epoch, gate_means_per_epoch):
    payload = {
        "epoch": epoch,
        "model_state": _cpu_state_dict(model),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "best_val_auc": best_val_auc,
        "best_state": best_state,
        "best_epoch": best_epoch,
        "gate_means_per_epoch": gate_means_per_epoch,
        "rng_state": _rng_state(),
    }
    _atomic_torch_save(payload, path)


def train_one_run(cfg_, seed, resume=True):
    set_seed(seed)
    model = TopoDistilModel(
        backbone_name=CONFIG["BACKBONE"],
        gate_active=cfg_["gate_active"], gate_form=cfg_.get("gate_form", "B"),
        use_L_attn=cfg_["use_L_attn"], use_residual=cfg_["use_residual"],
        homology_mode=cfg_["homology_mode"], shuffled_control=cfg_["shuffled_control"],
    ).to(DEVICE)

    train_ds = PCamTopoDataset(train_idx, "train", labels=train_labels, augment=True,
                                aug_mode="unaligned" if cfg_["unaligned_aug"] else "aligned",
                                seed=seed)
    train_loader = DataLoader(train_ds, batch_size=CONFIG["BATCH_SIZE"], shuffle=True,
                               num_workers=0, drop_last=True)
    val_ds = PCamTopoDataset(val_idx, "valid", labels=val_labels_sub, augment=False)
    val_loader = DataLoader(val_ds, batch_size=CONFIG["BATCH_SIZE"], shuffle=False, num_workers=0)
    test_ds = PCamTopoDataset(test_idx, "test", labels=test_labels_sub, augment=False)
    test_loader = DataLoader(test_ds, batch_size=CONFIG["BATCH_SIZE"], shuffle=False, num_workers=0)

    lam = CONFIG["LAMBDA"]

    # Build align_head (if this config uses L_attn) BEFORE the optimizer exists,
    # via one real dummy forward pass -- so every trainable param is in a known
    # param group from step 0. This removes the old runtime add_param_group()
    # call, which would otherwise silently desync from the LR scheduler.
    if cfg_["use_L_attn"]:
        model.eval()
        with torch.no_grad():
            dummy_img, dummy_amap, dummy_entropy, dummy_labels, _ = next(iter(train_loader))
            model(dummy_img.to(DEVICE), dummy_amap.to(DEVICE), dummy_entropy.to(DEVICE).float())
        model.train()

    steps_per_epoch = len(train_loader)
    optimizer = make_optimizer(model)
    scheduler = make_scheduler(optimizer, steps_per_epoch, CONFIG["EPOCHS"])

    rid = run_id(cfg_, seed)
    ckpt_path = f'{CONFIG["OUT_DIR"]}/{rid}_ckpt.pt'

    start_epoch = 0
    best_val_auc = -1.0
    best_state = None
    best_epoch = None
    gate_means_per_epoch = []

    if resume and os.path.exists(ckpt_path):
        try:
            ckpt = torch.load(ckpt_path, map_location=DEVICE)
            model.load_state_dict(ckpt["model_state"])
            optimizer.load_state_dict(ckpt["optimizer_state"])
            scheduler.load_state_dict(ckpt["scheduler_state"])
            best_val_auc = ckpt["best_val_auc"]
            best_state = ckpt["best_state"]
            best_epoch = ckpt["best_epoch"]
            gate_means_per_epoch = ckpt["gate_means_per_epoch"]
            _set_rng_state(ckpt["rng_state"])
            start_epoch = ckpt["epoch"] + 1
            print(f"[{rid}] resuming from checkpoint at epoch {start_epoch} "
                  f"(best_val_auc so far={best_val_auc:.4f} @ epoch {best_epoch})", flush=True)
        except Exception as resume_err:
            print(f"[{rid}] WARNING: found a checkpoint but failed to load it "
                  f"({resume_err}) -- restarting this run from epoch 0.", flush=True)
            start_epoch = 0
            best_val_auc = -1.0
            best_state = None
            best_epoch = None
            gate_means_per_epoch = []

    for epoch in range(start_epoch, CONFIG["EPOCHS"]):
        model.train()
        model.use_topology = True
        epoch_gate_means = []
        pbar = tqdm(train_loader, desc=f"{cfg_['name']} seed{seed} epoch{epoch}", leave=False)

        for step, (img, amap, entropy, labels_, idx) in enumerate(pbar):
            img, amap, entropy, labels_ = (img.to(DEVICE), amap.to(DEVICE),
                                            entropy.to(DEVICE).float(), labels_.to(DEVICE))
            logits, feat = model(img, amap, entropy)

            loss, ce, l_attn = compute_losses(model, logits, feat, amap, entropy, labels_, lam, cfg_)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            scheduler.step()

            if model.gate is not None and model.gate.last_gate_mean is not None:
                epoch_gate_means.append(model.gate.last_gate_mean)
            loss_val = float(loss.detach().cpu())
            cur_lr = optimizer.param_groups[0]["lr"]
            pbar.set_postfix(loss=loss_val, lr=cur_lr)

            # explicit flushed progress line -- visible even in batch/committed Kaggle runs
            # where stdout is only flushed at buffer boundaries / tqdm's \r updates get lost
            if step % 50 == 0:
                print(f"[{cfg_['name']} seed{seed}] epoch {epoch} step {step}/"
                      f"{len(train_loader)} loss={loss_val:.4f} lr={cur_lr:.2e}", flush=True)

        if epoch_gate_means:
            gate_means_per_epoch.append(float(np.mean(epoch_gate_means)))

        # --- per-epoch val AUC + best-checkpoint tracking (Section 11 fix #2) ---
        epoch_val = evaluate(model, val_loader, use_topology=False)
        is_best = epoch_val["auc"] > best_val_auc
        if is_best:
            best_val_auc = epoch_val["auc"]
            best_epoch = epoch
            best_state = _cpu_state_dict(model)
        model.train()  # evaluate() leaves the model in eval mode

        print(f"[{cfg_['name']} seed{seed}] epoch {epoch} DONE | "
              f"last_batch_loss={loss_val:.4f} | val_auc={epoch_val['auc']:.4f} "
              f"{'(new best)' if is_best else f'(best={best_val_auc:.4f} @ epoch {best_epoch})'}",
              flush=True)

        # epoch-level checkpoint -- if this process dies (OOM, time limit, kernel
        # restart) partway through the battery, re-running the same cell resumes
        # from here instead of losing the whole run.
        try:
            save_run_checkpoint(ckpt_path, epoch, model, optimizer, scheduler,
                                 best_val_auc, best_state, best_epoch, gate_means_per_epoch)
        except Exception as ckpt_err:
            print(f"[{rid}] WARNING: checkpoint save failed at epoch {epoch}: {ckpt_err}", flush=True)

        try:
            heartbeat_path = f'{CONFIG["OUT_DIR"]}/heartbeat.json'
            with open(f"{heartbeat_path}.tmp", "w") as hb:
                json.dump({
                    "run": rid,
                    "epoch": epoch,
                    "total_epochs": CONFIG["EPOCHS"],
                    "last_batch_loss": loss_val,
                    "val_auc_this_epoch": epoch_val["auc"],
                    "best_val_auc": best_val_auc,
                    "best_epoch": best_epoch,
                    "timestamp": time.time(),
                }, hb, indent=2)
            os.replace(f"{heartbeat_path}.tmp", heartbeat_path)
        except Exception as hb_err:
            print(f"heartbeat write failed: {hb_err}", flush=True)

    # --- final eval uses the BEST checkpoint, not whatever epoch 12 landed on ---
    if best_state is not None:
        model.load_state_dict(best_state)
    standard_val = evaluate(model, val_loader, use_topology=False)
    standard_test = evaluate(model, test_loader, use_topology=False)

    oracle_test = None
    if cfg_["oracle_eval"]:
        oracle_loader = DataLoader(
            PCamTopoDataset(test_idx[:CONFIG["ORACLE_SUBSET"]], "test",
                            labels=test_labels_sub[:CONFIG["ORACLE_SUBSET"]], augment=False),
            batch_size=CONFIG["BATCH_SIZE"], shuffle=False, num_workers=0,
        )
        oracle_test = evaluate(model, oracle_loader, use_topology=True, on_the_fly_topology=True)

    # Run is fully complete and the best weights are already baked into `model` /
    # about to be saved by the caller -- the resumable checkpoint has served its
    # purpose, so remove it rather than leaving a stale multi-hundred-MB file
    # per run sitting in OUT_DIR.
    try:
        if os.path.exists(ckpt_path):
            os.remove(ckpt_path)
    except Exception as cleanup_err:
        print(f"[{rid}] WARNING: could not remove checkpoint file {ckpt_path}: {cleanup_err}", flush=True)

    return model, {
        "standard_val": standard_val,
        "standard_test": standard_test,
        "oracle_test": oracle_test,
        "gate_mean_per_epoch": gate_means_per_epoch,
        "best_epoch": best_epoch,
        "best_val_auc_during_training": best_val_auc,
    }


## 12. Experimental matrix

| Configuration | Seeds | Purpose |
|---|---:|---|
| `mobilevit_baseline` | 3 | Plain MobileViT, no topology at all |
| `mobilevit_life` | 3 | MobileViT + LIFE (gate form B + L_attn), the transfer-question row |
| `mobilevit_shuffled_control` | 1 | LIFE with `A_proj` spatially scrambled -- tests whether gains come from spatial correctness or just extra parameters/gradient paths |
| `mobilevit_oracle` | 1 | LIFE with topology recomputed on the fly at test time -- upper bound, not the deployed configuration |
| `mobilevit_lattn_only_no_gate` | 3 | Distillation loss only, gate never applied (`F' = F`) -- tests whether NB2's row-4 pattern on RepViT (distillation-only beat full LIFE) transfers |

11 runs total. The first 8 match Section 16 of `TopoDistil_bulletproof.md`; `mobilevit_lattn_only_no_gate` (3 runs) is a follow-up addition, not part of the original plan.

In [ ]:
BASE = dict(gate_active=False, gate_form="B", use_L_attn=False, use_residual=True,
            homology_mode="both", shuffled_control=False, unaligned_aug=False, oracle_eval=False)

def cfg(name, **overrides):
    c = dict(BASE); c.update(overrides); c["name"] = name
    return c

EXPERIMENT_MATRIX = [
    cfg("mobilevit_baseline"),
    cfg("mobilevit_life", gate_active=True, use_L_attn=True),
    cfg("mobilevit_shuffled_control", gate_active=True, use_L_attn=True, shuffled_control=True),
    cfg("mobilevit_oracle", gate_active=True, use_L_attn=True, oracle_eval=True),
    cfg("mobilevit_lattn_only_no_gate", use_L_attn=True),  # new: gate-free distillation-only
]

SEEDS_MULTI = {"mobilevit_baseline", "mobilevit_life", "mobilevit_lattn_only_no_gate"}

run_plan = []
for c in EXPERIMENT_MATRIX:
    seeds = CONFIG["SEEDS_CORE"] if c["name"] in SEEDS_MULTI else [CONFIG["SEED_SINGLE"]]
    for s in seeds:
        run_plan.append((c, s))

print(f"Total planned runs: {len(run_plan)}")
for c, s in run_plan:
    print(f"  {c['name']:28s} seed={s}")


## 13. Run the battery (resumable)

In [ ]:
# run_id() is defined in Section 11 (used by train_one_run's checkpoint path too)

def _atomic_json_dump(obj, path):
    """Same rationale as the model checkpoints: write-then-rename so a killed
    process can never leave a half-written, corrupt manifest -- which would
    otherwise lose the resumability of every previously-completed run, not
    just the one in flight."""
    tmp_path = path + ".tmp"
    with open(tmp_path, "w") as f:
        json.dump(obj, f, indent=2)
    os.replace(tmp_path, path)


manifest_path = f'{CONFIG["OUT_DIR"]}/run_manifest.json'
manifest = json.load(open(manifest_path)) if os.path.exists(manifest_path) else {}

for c, seed in run_plan:
    rid = run_id(c, seed)
    if manifest.get(rid, {}).get("status") == "done":
        continue

    print(f"\n=== Running {rid} ===", flush=True)
    t0 = time.time()
    try:
        model, metrics = train_one_run(c, seed)
    except Exception as e:
        print(f"FAILED {rid}: {e}", flush=True)
        sys.stdout.flush()
        manifest[rid] = {"status": "failed", "error": str(e)}
        _atomic_json_dump(manifest, manifest_path)
        # Note: train_one_run saves an epoch-level checkpoint after every
        # completed epoch, so re-running this cell will resume this run from
        # its last completed epoch instead of starting over from epoch 0.
        continue

    elapsed = time.time() - t0
    metrics["elapsed_sec"] = elapsed
    metrics["config"] = c
    metrics["seed"] = seed
    metrics["backbone"] = CONFIG["BACKBONE"]

    _atomic_json_dump(metrics, f'{CONFIG["OUT_DIR"]}/{rid}_metrics.json')

    if c["name"] in CONFIG["SAVE_WEIGHTS_FOR"]:
        weights_path = f'{CONFIG["OUT_DIR"]}/{rid}_weights.pt'
        tmp_weights_path = weights_path + ".tmp"
        torch.save(model.state_dict(), tmp_weights_path)
        os.replace(tmp_weights_path, weights_path)

    manifest[rid] = {"status": "done", "elapsed_sec": elapsed}
    _atomic_json_dump(manifest, manifest_path)

    del model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    print(f"Done {rid} in {elapsed/60:.1f} min | "
          f"val AUC={metrics['standard_val']['auc']:.3f} "
          f"test AUC={metrics['standard_test']['auc']:.3f} "
          f"(best epoch={metrics['best_epoch']})", flush=True)

print("\nBattery complete." if all(
    manifest.get(run_id(c, s), {}).get("status") == "done" for c, s in run_plan
) else "\nSome runs incomplete -- re-run this cell to resume.", flush=True)


## 14. Reference: backbone size / latency / FLOPs

Not recomputed -- NB3 already measured this for `CONFIG["BACKBONE"]`. Pulled straight from its summary so NB5 has one consistent source for these numbers instead of two (possibly slightly different, if run on a different GPU) copies.

In [ ]:
backbone_reference = {
    "backbone": CONFIG["BACKBONE"],
    "params": nb3_summary["param_counts"].get(CONFIG["BACKBONE"]),
    "flops": nb3_summary.get("flops", {}).get(CONFIG["BACKBONE"]),
    "flops_method": nb3_summary.get("flops_method", {}).get(CONFIG["BACKBONE"]),
    "inference_profile": nb3_summary.get("inference_profile"),
    "source": "NB3 mobilevit_validation_summary.json (not recomputed here)",
}
with open(f'{CONFIG["OUT_DIR"]}/backbone_reference.json', "w") as f:
    json.dump(backbone_reference, f, indent=2)
backbone_reference


## 15. Package checkpoint bundle for NB5

In [ ]:
with open(f'{CONFIG["OUT_DIR"]}/config_used.json', "w") as f:
    json.dump({k: (list(v) if isinstance(v, set) else v) for k, v in CONFIG.items()}, f, indent=2)

print("MobileViT + LIFE output:")
for f in sorted(Path(CONFIG["OUT_DIR"]).iterdir()):
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name:45s} {size_mb:10.2f} MB")

print()
print("Next: Save Version. In NB5, add this notebook's output (mobilevit_life_results) as an")
print("input alongside NB2's (repvit_results) to load every *_metrics.json + run_manifest.json")
print("from both backbones, compute Delta_RepViT vs Delta_MobileViT, and run the")
print("cross-backbone significance testing / plots / report.")
